In [ ]:
# ==========================================
#  - READ ACTIVITY LOG DATA
# ==========================================

import pandas as pd

# CSV file ka path (r prefix lagao - raw string)
file_path = r"C:\Users\acer\Mini Project workshop\data\activitylogs.csv"

# CSV ko DataFrame mein load karna
df = pd.read_csv(file_path)

# Data successfully load hua ya nahi
print("Activity logs loaded successfully!")

# Number of rows aur columns
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Complete dataset display
print("\nActivity Logs:")
display(df)

Activity logs loaded successfully!
Rows: 10
Columns: 5

Columns:
['timestamp', 'module', 'event_type', 'status', 'duration_ms']

Activity Logs:


,timestamp,module,event_type,status,duration_ms
0,2026-10-07 10:01:00,Login,Login,Success,120
1,2026-10-07 10:02:00,Login,Login,Success,150
2,2026-10-07 10:03:00,Asset,Search,Success,230
3,2026-10-07 10:04:00,Asset,Search,Success,210
4,2026-10-07 10:05:00,Checkout,Checkout,Error,850
5,2026-10-07 10:06:00,Checkout,Checkout,Error,920
6,2026-10-07 10:07:00,Reports,Generate,Success,310
7,2026-10-07 10:08:00,Reports,Generate,Success,280
8,2026-10-07 10:09:00,Checkout,Checkout,Error,780
9,2026-10-07 10:10:00,Login,Login,Error,180


In [6]:
# ==========================================
# - DATA VALIDATION
# STEP 4.1 - CHECK REQUIRED COLUMNS
# ==========================================

required_columns = [
    "timestamp",
    "module",
    "event_type",
    "status",
    "duration_ms"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    print(" Missing columns:", missing_columns)
else:
    print(" All required columns are present.")

 All required columns are present.


In [7]:
# Check missing values in each column

missing_values = df.isnull().sum()

print("Missing values:")
print(missing_values)

Missing values:
timestamp      0
module         0
event_type     0
status         0
duration_ms    0
dtype: int64


In [8]:
# Check duplicate rows

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [9]:
# Convert timestamp to datetime
# Invalid timestamps will become NaT instead of crashing

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print("Invalid timestamps:", df["timestamp"].isnull().sum())

Invalid timestamps: 0


In [10]:
# Convert duration into numeric values
# Invalid values become NaN

df["duration_ms"] = pd.to_numeric(
    df["duration_ms"],
    errors="coerce"
)

print("Invalid duration values:", df["duration_ms"].isnull().sum())

Invalid duration values: 0


In [11]:
# ==========================================
# VALIDATION SUMMARY
# ==========================================

print("========== VALIDATION SUMMARY ==========")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Invalid timestamps:", df["timestamp"].isnull().sum())
print("Invalid durations:", df["duration_ms"].isnull().sum())

print("========================================")

========== VALIDATION SUMMARY ==========
Rows: 10
Columns: 5
Missing values: 0
Duplicate rows: 0
Invalid timestamps: 0
Invalid durations: 0


In [12]:
# ==========================================
#  - ANALYTICS ENGINE
# STEP 5.1 - TOTAL EVENTS
# ==========================================

total_events = len(df)

print("Total Events:", total_events)

Total Events: 10


In [13]:
# Count total errors

total_errors = (df["status"] == "Error").sum()

print("Total Errors:", total_errors)

Total Errors: 4


In [14]:
#  overall error rate

error_rate = (total_errors / total_events) * 100

print("Overall Error Rate:", round(error_rate, 2), "%")

Overall Error Rate: 40.0 %


In [15]:
# average duration

average_duration = df["duration_ms"].mean()

print(
    "Average Duration:",
    round(average_duration, 2),
    "ms"
)

Average Duration: 403.0 ms


In [16]:
#  median duration

median_duration = df["duration_ms"].median()

print(
    "Median Duration:",
    round(median_duration, 2),
    "ms"
)

Median Duration: 255.0 ms


In [17]:
# ==========================================
# MODULE-WISE EVENT COUNT
# ==========================================

module_event_count = (
    df.groupby("module")
      .size()
      .reset_index(name="total_events")
)

module_event_count

,module,total_events
0,Asset,2
1,Checkout,3
2,Login,3
3,Reports,2


In [18]:
# ==========================================
# MODULE-WISE ERROR COUNT
# ==========================================

module_error_count = (
    df[df["status"] == "Error"]
    .groupby("module")
    .size()
    .reset_index(name="total_errors")
)

module_error_count

,module,total_errors
0,Checkout,3
1,Login,1


In [19]:
# ==========================================
# MODULE-WISE ERROR RATE
# ==========================================

module_stats = (
    df.groupby("module")
      .agg(
          total_events=("status", "size"),
          total_errors=("status", lambda x: (x == "Error").sum()),
          average_duration_ms=("duration_ms", "mean"),
          median_duration_ms=("duration_ms", "median")
      )
      .reset_index()
)

module_stats["error_rate"] = (
    module_stats["total_errors"]
    / module_stats["total_events"]
) * 100

module_stats["error_rate"] = module_stats["error_rate"].round(2)

module_stats["average_duration_ms"] = (
    module_stats["average_duration_ms"].round(2)
)

module_stats["median_duration_ms"] = (
    module_stats["median_duration_ms"].round(2)
)

module_stats

,module,total_events,total_errors,average_duration_ms,median_duration_ms,error_rate
0,Asset,2,0,220.0,220.0,0.00
1,Checkout,3,3,850.0,850.0,100.00
2,Login,3,1,150.0,150.0,33.33
3,Reports,2,0,295.0,295.0,0.00


In [20]:
# ==========================================
#  - UNUSUAL PATTERN DETECTION
# STEP 6.1 - DEFINE THRESHOLD
# ==========================================

ERROR_RATE_THRESHOLD = 5.0

print("Error Rate Threshold:", ERROR_RATE_THRESHOLD, "%")

Error Rate Threshold: 5.0 %


In [21]:
# ==========================================
# STEP 6.2 - FLAG MODULES
# ==========================================

module_stats["status"] = module_stats["error_rate"].apply(
    lambda rate: "Flagged" 
    if rate > ERROR_RATE_THRESHOLD 
    else "Normal"
)

module_stats

,module,total_events,total_errors,average_duration_ms,median_duration_ms,error_rate,status
0,Asset,2,0,220.0,220.0,0.00,Normal
1,Checkout,3,3,850.0,850.0,100.00,Flagged
2,Login,3,1,150.0,150.0,33.33,Flagged
3,Reports,2,0,295.0,295.0,0.00,Normal


In [22]:
# ==========================================
# STEP 6.3 - FLAGGED MODULES
# ==========================================

flagged_modules = module_stats[
    module_stats["status"] == "Flagged"
]

print("Flagged Modules:")
display(flagged_modules)

Flagged Modules:


,module,total_events,total_errors,average_duration_ms,median_duration_ms,error_rate,status
1,Checkout,3,3,850.0,850.0,100.00,Flagged
2,Login,3,1,150.0,150.0,33.33,Flagged


In [23]:
# Count flagged modules

flagged_count = len(flagged_modules)

print("Total Flagged Modules:", flagged_count)

Total Flagged Modules: 2
